# 04 — Baseline LSTM Training (4 variants)

This notebook trains **four baseline LSTMs** — one per variant — by invoking `src/train_LSTM_baseline.py` in sequence with variant-specific feature sets and output prefixes:

| Variant | Features | n_feats | Output checkpoint |
|---|---|---|---|
| **O** | stationary only | 5 | `lstm_baseline_O.pt` |
| **A** | stationary + AAII sentiment | 7 | `lstm_baseline.pt` *(default prefix preserved)* |
| **H** | A + HMM posterior `p_volatile` | 8 | `lstm_baseline_H.pt` |
| **B** | A + VIX family (vix, log_change, 3m−spot) | 10 | `lstm_baseline_B.pt` |

Each variant runs the same `train_LSTM_baseline.py` pipeline:

1. Load the train / val / test splits from `data/processed/`.
2. Select input features via CLI `--features`.
3. Log-transform the target `realized_vol_21d` for training.
4. Fit `StandardScaler` on the train features.
5. Run Optuna TPE (search space in `config.LSTM_SEARCH_SPACE`) scored on val MSE in raw (inverse-log) scale.
6. Retrain with best params; evaluate on test.
7. Save `{prefix}.pt` + `{prefix}_scaler.joblib`.

**Prerequisites:** `nb 01` must have produced `train/val/test.parquet`; `nb 03` must have injected variant-A's `p_volatile` column into those parquets (required for variant H). A guard cell below verifies this.

**Training-window asymmetry (intentional, documented):**
- Variants O, A, H: ~3,690 train rows (sentiment + p_volatile have negligible NaN at the start).
- Variant B: ~2,030 train rows — bottlenecked by `vix3m_minus_vix` starting 2007-12 (FRED's VXVCLS inception). Documented as a limitation in the paper.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path().resolve().parent
SCRIPT = REPO_ROOT / "src" / "train_LSTM_baseline.py"
assert SCRIPT.exists(), f"Missing: {SCRIPT}"
sys.path.insert(0, str(REPO_ROOT))

import config

print("Repo root:", REPO_ROOT)
print("Script   :", SCRIPT)


## Prerequisite check

Confirms the parquets have the features all four variants need. If `p_volatile` is missing, nb 03's injection cell hasn't run yet — running variant H will fail.


In [ ]:
train_df = pd.read_parquet(config.DATA_PROCESSED / "train.parquet")

required_cols = (
    set(config.LSTM_VARIANT_O_FEATURES)
    | set(config.LSTM_VARIANT_A_FEATURES)
    | set(config.LSTM_VARIANT_H_FEATURES)
    | set(config.LSTM_VARIANT_B_FEATURES)
    | {config.LSTM_TARGET}
)
missing = sorted(required_cols - set(train_df.columns))
if missing:
    raise RuntimeError(
        f"train.parquet is missing columns required by one of the variants: {missing}. "
        f"Run nb 01 (feature engineering) and nb 03 (p_volatile injection cell 10c) first."
    )

# Concrete NaN counts that affect training-window size per variant:
def _trainable_rows(df, feats):
    return int(df[list(feats) + [config.LSTM_TARGET]].dropna().shape[0])

summary = pd.DataFrame({
    "variant": ["O", "A", "H", "B"],
    "n_features": [
        len(config.LSTM_VARIANT_O_FEATURES),
        len(config.LSTM_VARIANT_A_FEATURES),
        len(config.LSTM_VARIANT_H_FEATURES),
        len(config.LSTM_VARIANT_B_FEATURES),
    ],
    "trainable_rows": [
        _trainable_rows(train_df, config.LSTM_VARIANT_O_FEATURES),
        _trainable_rows(train_df, config.LSTM_VARIANT_A_FEATURES),
        _trainable_rows(train_df, config.LSTM_VARIANT_H_FEATURES),
        _trainable_rows(train_df, config.LSTM_VARIANT_B_FEATURES),
    ],
})
print("Per-variant trainable rows (after dropping NaN on features + target):")
print(summary.to_string(index=False))


## Define variant sweep

Edit `VARIANTS` below if you need to skip a variant (e.g., for a partial rerun). The default runs all four.


In [ ]:
VARIANTS = [
    {"name": "O", "features": config.LSTM_VARIANT_O_FEATURES, "output_prefix": "lstm_baseline_O"},
    {"name": "A", "features": config.LSTM_VARIANT_A_FEATURES, "output_prefix": "lstm_baseline"},
    {"name": "H", "features": config.LSTM_VARIANT_H_FEATURES, "output_prefix": "lstm_baseline_H"},
    {"name": "B", "features": config.LSTM_VARIANT_B_FEATURES, "output_prefix": "lstm_baseline_B"},
]
for v in VARIANTS:
    print(f"  {v['name']}: {v['output_prefix']}.pt  ({len(v['features'])} feats)")


## Training sweep

Runs each variant sequentially, streaming the subprocess's stdout + stderr live. If a variant fails, the error is logged and the loop continues with the next variant so you don't lose earlier progress.

Each variant takes roughly the same wall time as a single-variant run (Optuna trials × tune-epochs + final retrain). Expected: 10-20 min per variant on CPU, faster on GPU.


In [ ]:
variant_outputs = {}

for v in VARIANTS:
    name, features, prefix = v["name"], v["features"], v["output_prefix"]
    ARGS = [
        "--features", *features,
        "--output-prefix", prefix,
    ]
    cmd = [sys.executable, "-u", str(SCRIPT), *ARGS]

    print("=" * 80)
    print(f"[Variant {name}]  prefix={prefix}  features={features}")
    print("Command:", " ".join(cmd))
    print("-" * 80)

    captured_lines = []
    try:
        proc = subprocess.Popen(
            cmd,
            cwd=str(REPO_ROOT),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            captured_lines.append(line)
        return_code = proc.wait()
        if return_code != 0:
            print(f"\n[Variant {name}]  FAILED with return code {return_code}")
            variant_outputs[name] = {"status": "failed", "return_code": return_code, "output": "".join(captured_lines)}
            continue
    except Exception as exc:
        print(f"\n[Variant {name}]  EXCEPTION: {exc}")
        variant_outputs[name] = {"status": "exception", "error": str(exc), "output": "".join(captured_lines)}
        continue

    variant_outputs[name] = {"status": "ok", "output": "".join(captured_lines)}
    print(f"\n[Variant {name}]  ok — artifacts saved under prefix '{prefix}'.")
    print()


## Parse and aggregate results

Extract each variant's JSON results block (emitted by `train_LSTM_baseline.py` under `=== Baseline LSTM results ===`) and build a comparison table.


In [ ]:
marker = "=== Baseline LSTM results ==="

results_per_variant = {}
for name, info in variant_outputs.items():
    if info.get("status") != "ok":
        print(f"  [{name}] skipped — status={info.get('status')}")
        continue
    text = info["output"]
    idx = text.find(marker)
    if idx == -1:
        print(f"  [{name}] WARNING: results marker not found; training may have been truncated.")
        continue
    json_blob = text[idx + len(marker):].strip()
    try:
        results_per_variant[name] = json.loads(json_blob)
    except json.JSONDecodeError as exc:
        print(f"  [{name}] JSON parse failed: {exc}")
        continue

# Comparison DataFrame
rows = []
for name, r in results_per_variant.items():
    rows.append({
        "variant":        name,
        "n_features":     len(r["features"]),
        "best_val_mse":   r["best_val_mse_raw"],
        "retrain_val_mse": r["retrained_val_mse_raw"],
        "test_MSE":       r["test_metrics"]["MSE"],
        "test_RMSE":      r["test_metrics"]["RMSE"],
        "test_MAE":       r["test_metrics"]["MAE"],
        "test_MAPE":      r["test_metrics"].get("MAPE"),
        "seq_len":        r["best_params"]["seq_len"],
        "hidden_size":    r["best_params"]["hidden_size"],
        "n_layers":       r["best_params"]["n_layers"],
        "lr":             r["best_params"]["lr"],
        "n_trials":       r["n_trials"],
    })

headline = pd.DataFrame(rows).set_index("variant")
print("Per-variant baseline LSTM comparison:")
headline


## Summary (fill in after execution)

Use the headline table above for the paper's §4.3 baseline LSTM comparison. Key framing for the paper:

- Does variant A beat variant O on test MSE? → quantifies the value of AAII sentiment features alone.
- Does variant H beat variant A? → tests "regime-as-feature" as a single-LSTM architectural alternative to the regime-split ensemble (variant A's ensemble is evaluated in nb 05 / nb 06).
- Does variant B beat variant A? → tests forward-looking VIX features (with the caveat of a shorter training window).

Significance testing (Diebold-Mariano) for each pair happens in the variant-comparison notebook (nb 07), not here — this notebook only produces per-variant point-estimate metrics.

**Training-window caveat:** variant B's smaller training window (~2,030 rows vs ~3,690 for O/A/H) means any metric gap is entangled with sample-size differences, not just feature differences. Report this explicitly in the paper.


## Saved artifacts

| Variant | Checkpoint | Scaler |
|---|---|---|
| O | `models/lstm_baseline_O.pt` | `models/lstm_baseline_O_scaler.joblib` |
| A | `models/lstm_baseline.pt` | `models/lstm_baseline_scaler.joblib` |
| H | `models/lstm_baseline_H.pt` | `models/lstm_baseline_H_scaler.joblib` |
| B | `models/lstm_baseline_B.pt` | `models/lstm_baseline_B_scaler.joblib` |

Each `.pt` contains state_dict, hyperparameters, feature list, target, and regime-state metadata. Reload pattern:

```python
import torch, joblib
from src.lstm_model import LSTMRegressor
ckpt = torch.load('../models/lstm_baseline_O.pt', weights_only=False, map_location='cpu')
model = LSTMRegressor(input_size=ckpt['n_features'], **{k: ckpt['hyperparameters'][k] for k in ('hidden_size','n_layers','dropout')})
model.load_state_dict(ckpt['state_dict'])
scaler = joblib.load('../models/lstm_baseline_O_scaler.joblib')
```

All four checkpoints feed into `src/ensemble.py` (nb 05 → `variant_outputs`) for the soft-probability ensemble step.
